In [ ]:
!pip install scikit-learn==1.5.2 --upgrade -q
!pip install optuna catboost xgboost lightgbm --upgrade -q

import pandas as pd
import numpy as np
import optuna
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
from google.colab import files

# 1. Load Data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

def clean_data_robust(df_train, df_test):
    len_train = len(df_train)
    df = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

    df[['Group', 'GroupNum']] = df['PassengerId'].str.split('_', expand=True)
    df['Surname'] = df['Name'].str.split().str[-1]

    cols_shared = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
    for col in cols_shared:
        df[col] = df.groupby('Group')[col].transform(lambda x: x.ffill().bfill())

    df['Cabin'] = df.groupby('Group')['Cabin'].transform(lambda x: x.ffill().bfill())
    cabin_split = df['Cabin'].str.split('/', expand=True)
    df['CabinDeck'] = cabin_split[0].fillna(df['CabinDeck'].mode()[0] if 'CabinDeck' in df else 'F')
    df['CabinNum'] = pd.to_numeric(cabin_split[1], errors='coerce').fillna(df['CabinNum'].median() if 'CabinNum' in df else 0)
    df['CabinSide'] = cabin_split[2].fillna(df['CabinSide'].mode()[0] if 'CabinSide' in df else 'S')

    amenities = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df[amenities] = df[amenities].fillna(0)
    df['TotalSpend'] = df[amenities].sum(axis=1)

    df.loc[df['TotalSpend'] > 0, 'CryoSleep'] = False
    df['CryoSleep'] = df['CryoSleep'].fillna(False).astype(int)
    df['VIP'] = df['VIP'].fillna(False).astype(int)

    # Features
    df['DeckSide'] = df['CabinDeck'] + df['CabinSide']
    df['FamilySize'] = df['Surname'].map(df['Surname'].value_counts()).fillna(1)
    df['GroupSize'] = df['Group'].map(df['Group'].value_counts()).fillna(1)

    for col in amenities + ['TotalSpend']:
        df[col] = np.log1p(df[col])

    cat_cols = ['HomePlanet', 'Destination', 'CabinDeck', 'CabinSide', 'DeckSide']
    df = pd.get_dummies(df, columns=cat_cols, dummy_na=False, dtype=int)

    cols_drop = ['PassengerId', 'Cabin', 'Name', 'Group', 'GroupNum', 'Surname', 'Transported']
    df = df.drop(columns=[c for c in cols_drop if c in df.columns], errors='ignore')

    return df.iloc[:len_train], df.iloc[len_train:]

print("🧹 Processing Data...")
X, X_test_final = clean_data_robust(train, test)
y = train['Transported'].astype(int)

# 3. Tuning Function
def tune_model(model_name, X_data, y_data, n_trials=50):
    def objective(trial):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        if model_name == 'xgb':
            params = {
                'tree_method': 'hist', 'device': 'cuda', 'objective': 'binary:logistic', 'eval_metric': 'logloss',
                'use_label_encoder': False, 'random_state': 42,
                'n_estimators': trial.suggest_int('n_estimators', 400, 800),
                'max_depth': trial.suggest_int('max_depth', 4, 9),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
            }
            model = xgb.XGBClassifier(**params)

        elif model_name == 'cat':
            params = {
                'task_type': 'CPU', 'verbose': 0, 'random_seed': 42,
                'iterations': trial.suggest_int('iterations', 500, 1000),
                'depth': trial.suggest_int('depth', 4, 9),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10)
            }
            model = CatBoostClassifier(**params)

        elif model_name == 'lgbm':
            params = {
                'device': 'cpu', 'verbose': -1, 'random_state': 42,
                'n_estimators': trial.suggest_int('n_estimators', 400, 800),
                'max_depth': trial.suggest_int('max_depth', 5, 12),
                'num_leaves': trial.suggest_int('num_leaves', 20, 80),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True)
            }
            model = lgb.LGBMClassifier(**params)

        return cross_val_score(model, X_data, y_data, cv=cv, scoring='accuracy').mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

print("\n🔥 Tuning Best Params...")
best_params = {}
for m in ['xgb', 'cat', 'lgbm']:
    print(f"   -> Tuning {m}...")
    best_params[m] = tune_model(m, X, y)

print("\n🤝 Building Multi-Seed Ensemble (Reducing Variance)...")

estimators = []

seeds = [42, 2024, 999, 2025, 777]

for s in seeds:
    model = xgb.XGBClassifier(**best_params['xgb'], tree_method='hist', device='cuda', eval_metric='logloss', use_label_encoder=False, random_state=s)
    estimators.append((f'xgb_{s}', model))

for s in seeds:
    model = CatBoostClassifier(**best_params['cat'], task_type='CPU', verbose=0, random_seed=s)
    estimators.append((f'cat_{s}', model))

for s in seeds:
    model = lgb.LGBMClassifier(**best_params['lgbm'], device='cpu', verbose=-1, random_state=s)
    estimators.append((f'lgbm_{s}', model))

rf_model = RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=2, n_jobs=-1, random_state=42)
estimators.append(('rf', rf_model))

weights = [2/5]*5 + [2/5]*5 + [1/5]*5 + [1]

voting_clf = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=weights
)

print("🚀 Training Final Ensemble (Total 10 Models)...")
voting_clf.fit(X, y)

# ... (ต่อจากโค้ดเดิมของคุณ หลังจากบรรทัด voting_clf.fit(X, y)) ...

print("\n🔮 1. Generating Pseudo-labels (Threshold > 0.90)...")

# 1. หาความน่าจะเป็น (Probability) ของชุด Test ทั้งหมด
probs = voting_clf.predict_proba(X_test_final)

# probs จะเป็น array 2 คอลัมน์ [prob_class_0, prob_class_1]
# เราต้องการแถวที่มั่นใจมากๆ คือ max(prob) > 0.90
max_probs = probs.max(axis=1)
high_conf_mask = max_probs > 0.90

# 2. คัดเลือกข้อมูล (X) และสร้างฉลาก (y) จากสิ่งที่โมเดลทำนาย
X_pseudo = X_test_final[high_conf_mask]
y_pseudo = voting_clf.predict(X_pseudo) # เอาค่า 0 หรือ 1 ที่ทำนายได้มาเป็น Label

print(f"   -> Found {len(X_pseudo)} samples with confidence > 90%")
print(f"   -> Adding these to training data...")

# 3. รวมร่าง (Augment Data)
X_augmented = pd.concat([X, X_pseudo], axis=0)
# y เดิมเป็น Series, y_pseudo เป็น array ต้องแปลงให้เข้ากัน
y_pseudo_series = pd.Series(y_pseudo, index=X_pseudo.index)
y_augmented = pd.concat([y, y_pseudo_series], axis=0)

print(f"   -> New Training Size: {len(X_augmented)} (Original: {len(X)})")

# 4. Retrain Model (เทรนใหม่ด้วยข้อมูลที่เยอะขึ้น)
print("\n🚀 2. Retraining Ensemble with Pseudo-labeled Data...")
# หมายเหตุ: การ Retrain VotingClassifier จะเทรนทุก Sub-model ข้างในใหม่ทั้งหมด
voting_clf.fit(X_augmented, y_augmented)

# 5. Final Prediction
print("\n📝 Creating Final Submission (Refined)...")
final_pred_refined = voting_clf.predict(X_test_final)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': final_pred_refined.astype(bool)
})

filename = 'submission_pseudo_label_90.csv'
submission.to_csv(filename, index=False)
print(f"🎉 MISSION ACCOMPLISHED! Download: {filename}")
files.download(filename)